### Ingest races.csv file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../common/env_config

In [0]:
%run ../common/bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

#### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

races_schema = StructType([
    StructField('season',   IntegerType()),
    StructField("round",    IntegerType()),
    StructField("url",      StringType()),
    StructField("raceName", StringType()),
    StructField("date",     DateType()),
    StructField("circuitId", StringType())
])

In [0]:
races_df = (
    spark.read
         .format('csv')
         .option('header', 'true')
#         .option('inferSchema', 'true')
         .option('mode', 'FAILFAST')
         .schema(races_schema)
         .load(source_file)
)

In [0]:
display(races_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
from pyspark.sql import functions as F

races_final_df = add_ingestion_metadata(races_df)
                     

In [0]:
display(races_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
write_to_bronze (
    input_df = races_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
%sql
SELECT * FROM formula_one_incr.bronze.races